# Tokenizer Examples 1

>A tokenizer is in charge of preparing the inputs for a model. The library contains tokenizers for all the models. Most of the tokenizers are available in two flavors: a full python implementation and a “Fast” implementation based on the Rust library 🤗 Tokenizers. The “Fast" implementations allows:
>
> * a significant speed-up in particular when doing batched tokenization and
> * additional methods to map between the original string (character and words) and the token space (e.g. getting the index of the token comprising a given character or the span of characters corresponding to a given token).
>
>The base classes [PreTrainedTokenizer](https://huggingface.co/docs/transformers/v4.39.0/en/main_classes/tokenizer#transformers.PreTrainedTokenizer) and [PreTrainedTokenizerFast](https://huggingface.co/docs/transformers/v4.39.0/en/main_classes/tokenizer#transformers.PreTrainedTokenizerFast) implement the common methods for encoding string inputs in model inputs (see below) and instantiating/saving python and “Fast” tokenizers either from a local file or directory or from a pretrained tokenizer provided by the library (downloaded from HuggingFace’s AWS S3 repository). They both rely on [PreTrainedTokenizerBase](https://huggingface.co/docs/transformers/v4.39.0/en/internal/tokenization_utils#transformers.PreTrainedTokenizerBase) that contains the common methods, and [SpecialTokensMixin](https://huggingface.co/docs/transformers/v4.39.0/en/internal/tokenization_utils#transformers.SpecialTokensMixin).
>
>[PreTrainedTokenizer](https://huggingface.co/docs/transformers/v4.39.0/en/main_classes/tokenizer#transformers.PreTrainedTokenizer) and [PreTrainedTokenizerFast](https://huggingface.co/docs/transformers/v4.39.0/en/main_classes/tokenizer#transformers.PreTrainedTokenizerFast) thus implement the main methods for using all the tokenizers:
>
> * Tokenizing (splitting strings in sub-word token strings), converting tokens strings to ids and back, and encoding/decoding (i.e., tokenizing and converting to integers).
> * Adding new tokens to the vocabulary in a way that is independent of the underlying structure (BPE, SentencePiece…).
> *Managing special tokens (like mask, beginning-of-sentence, etc.): adding them, assigning them to attributes in the tokenizer for easy access and making sure they are not split during tokenization.

Source: [https://huggingface.co/docs/transformers/main_classes/tokenizer](https://huggingface.co/docs/transformers/main_classes/tokenizer)

The tokenizer library is available at [https://github.com/huggingface/tokenizers](https://github.com/huggingface/tokenizers)

## Special Tokens

### Tokens [CLS] and [SEP]

BERT has been trained using the format: [CLS] sentence 1 [SEP] sentence 2 [SEP].

In other words, BERT prepends a [CLS] token (short for classification) to the start of each sentence. In essence, this can be thought of being like a start-of-sentence token.[SEP] token separates sentences.

In [56]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from pprint import pp

In [57]:
# Create a tokenizer that is compatible with the model "google-bert/bert-base-uncased"
checkpoint = "google-bert/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [58]:
# Print out information regarding the tokenizer
pp(tokenizer)

BertTokenizerFast(name_or_path='google-bert/bert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}


### Convert Tokens to IDS and Back to Tokens

In [59]:
# Tokenize the string "hello world" and print the tokens
tokens = tokenizer.tokenize("hello world")
print(tokens)

['hello', 'world']


In [60]:
# Convert tokens to indices and print the indices
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids)

[7592, 2088]


In [61]:
# Convert the indices back to tokens
tokenizer.convert_ids_to_tokens(ids)

['hello', 'world']

In [62]:
# Decode the indices, i.e. convert the ids back to a string
tokenizer.decode(ids)

'hello world'

In [63]:
# How are suffixes treated by this tokenizer?
# How are the words "go" and "going" converted into ids?
text = "I will go"
ids1 = tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text))
print(f"{text} = {ids1}")

text = "I am going"
ids2 = tokenizer.convert_tokens_to_ids(tokenizer.tokenize("I am going"))
print(f"{text} = {ids1}")

I will go = [1045, 2097, 2175]
I am going = [1045, 2097, 2175]


### Encode and Decode

In [64]:
# Encode the string. Tokenizes the string and converts it to indices and adds start, end and other tokens
ids = tokenizer.encode("hello world")
print(ids)

[101, 7592, 2088, 102]


In [65]:
tokenizer.decode(ids)

'[CLS] hello world [SEP]'

### Generate Input for The Model

In [66]:
model_inputs = tokenizer("hello world")
pp(model_inputs)

{'input_ids': [101, 7592, 2088, 102],
 'token_type_ids': [0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1]}


In [67]:
text_input = ["I like cats.", "Do you like cats too?"]
pp(tokenizer(text_input))

{'input_ids': [[101, 1045, 2066, 8870, 1012, 102],
               [101, 2079, 2017, 2066, 8870, 2205, 1029, 102]],
 'token_type_ids': [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0]],
 'attention_mask': [[1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1]]}


## Sequence Classification Example

Since we're using a BERT model, it's worthwhile explaining a few things about the model:

>BERT is a transformer model pretrained on a large corpus of English data in a self-supervised fashion. This means it was pretrained on the raw texts only, with no humans labeling them in any way (which is why it can use lots of publicly available data) with an automatic process to generate inputs and labels from those texts. More precisely, it was pretrained with two objectives:
>
> * Masked language modeling (MLM): taking a sentence, the model randomly masks 15% of the words in the input then run the entire masked sentence through the model and has to predict the masked words. This is different from traditional recurrent neural networks (RNNs) that usually see the words one after the other, or from autoregressive models like GPT which internally masks the future tokens. It allows the model to learn a bidirectional representation of the sentence.
>
> * Next sentence prediction (NSP): the models concatenates two masked sentences as inputs during pretraining. Sometimes they correspond to sentences that were next to each other in the original text, sometimes not. The model then has to predict if the two sentences were following each other or not.
>
>This way, the model learns an inner representation of the English language that can then be used to extract features useful for downstream tasks: if you have a dataset of labeled sentences, for instance, you can train a standard classifier using the features produced by the BERT model as inputs.

When we create the sequence classification model, we get a warning saying that the head of the model has not been trained, i.e. the results we get are meaningless at this stage. We need to train the head in order to get meaningful results.

In [68]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

In [69]:
# Let's print out some information regarding the model
pp(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [70]:
model_inputs = tokenizer("hello world", return_tensors='pt')

In [71]:
outputs = model(**model_inputs)
pp(outputs)

SequenceClassifierOutput(loss=None,
                         logits=tensor([[0.1912, 0.0797]], grad_fn=<AddmmBackward0>),
                         hidden_states=None,
                         attentions=None)


In [72]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=3)

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

In [73]:
# Let's print out some information regarding the model
pp(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [74]:
outputs = model(**model_inputs)
pp(outputs)

SequenceClassifierOutput(loss=None,
                         logits=tensor([[ 0.4872, -0.7823, -0.3912]], grad_fn=<AddmmBackward0>),
                         hidden_states=None,
                         attentions=None)


In [75]:
outputs.logits

tensor([[ 0.4872, -0.7823, -0.3912]], grad_fn=<AddmmBackward0>)

In [76]:
outputs.logits.detach().cpu().numpy()

array([[ 0.48723567, -0.782287  , -0.39115664]], dtype=float32)

In [77]:
# For batched inputs, depending on the version of 'transformers' library you are using, you might need to add the
# parameters padding=True and truncation=True if the output is in a tensor format. Tensors need to have same length.
text_input = ["I like cats.", "Do you like cats too?"]
model_inputs = tokenizer(text_input, return_tensors='pt', padding=True, truncation=True)

In [78]:
# print the model_inputs
pp(model_inputs)

{'input_ids': tensor([[ 101, 1045, 2066, 8870, 1012,  102,    0,    0],
        [ 101, 2079, 2017, 2066, 8870, 2205, 1029,  102]]),
 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0]]),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1]])}


In [79]:
# print the input_ids
print(model_inputs.input_ids)

tensor([[ 101, 1045, 2066, 8870, 1012,  102,    0,    0],
        [ 101, 2079, 2017, 2066, 8870, 2205, 1029,  102]])


In [80]:
# print the attention mask, i.e. which tokens are used
print(model_inputs.attention_mask)

tensor([[1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1]])


In [81]:
# Do inference using the batch model_input
outputs = model(**model_inputs)

In [82]:
pp(outputs)

SequenceClassifierOutput(loss=None,
                         logits=tensor([[ 0.4968, -1.0804, -0.4048],
        [ 0.4519, -1.0921, -0.4134]], grad_fn=<AddmmBackward0>),
                         hidden_states=None,
                         attentions=None)
